In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json

import sys

sys.path.append("../")

##################################################################
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3,4,5,6,7"
##################################################################

import logging
from src.utils import logging_utils
from src.utils import env_utils

logger = logging.getLogger(__name__)

logging.basicConfig(
    level=logging.DEBUG,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)

import torch
import transformers

logger.info(f"{torch.__version__=}, {torch.version.cuda=}")
logger.info(
    f"{torch.cuda.is_available()=}, {torch.cuda.device_count()=}, {torch.cuda.get_device_name()=}"
)
logger.info(f"{transformers.__version__=}")

/disk/u/arnab/miniconda3/envs/connection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-11-17 18:51:53 __main__ INFO     torch.__version__='2.9.0+cu128', torch.version.cuda='12.8'
2025-11-17 18:51:54 __main__ INFO     torch.cuda.is_available()=True, torch.cuda.device_count()=8, torch.cuda.get_device_name()='NVIDIA A100-SXM4-80GB'
2025-11-17 18:51:54 __main__ INFO     transformers.__version__='4.57.1'


## Loading the LM

In [3]:
from src.utils.training_utils import get_device_map

# model_key = "meta-llama/Llama-3.2-3B"
# model_key = "meta-llama/Llama-3.1-8B-Instruct"
# model_key = "meta-llama/Llama-3.3-70B-Instruct"
# model_key = "meta-llama/Llama-3.1-405B-Instruct"

# model_key = "google/gemma-2-9b-it"
model_key = "google/gemma-2-27b-it"

# model_key = "openai/gpt-oss-20b"

# model_key = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# model_key = "allenai/OLMo-2-1124-7B-Instruct"
# model_key = "allenai/OLMo-7B-0424-hf"

# model_key = "Qwen/Qwen2-7B"
# model_key = "Qwen/Qwen2.5-14B-Instruct"
# model_key = "Qwen/Qwen2.5-32B-Instruct"
# model_key = "Qwen/Qwen2.5-72B-Instruct"

# model_key = "Qwen/Qwen3-1.7B"
# model_key = "Qwen/Qwen3-4B"
# model_key = "Qwen/Qwen3-8B"
# model_key = "Qwen/Qwen3-14B"
# model_key = "Qwen/Qwen3-32B"

# device_map = get_device_map(model_key, 30, n_gpus=8)
# device_map

2025-11-17 18:52:02 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)
2025-11-17 18:52:02 git.cmd DEBUG    Popen(['git', 'version'], cwd=/disk/u/arnab/Codes/Projects/filter/notebooks, stdin=None, shell=False, universal_newlines=False)


In [4]:
from src.models import ModelandTokenizer

# from transformers import BitsAndBytesConfig

mt = ModelandTokenizer(
    model_key=model_key,
    dtype=torch.bfloat16,
    # device_map=device_map,
    device_map="auto",
    # quantization_config = BitsAndBytesConfig(
    #     # load_in_4bit=True
    #     load_in_8bit=True
    # )
    attn_implementation="eager",
)

2025-11-17 18:52:08 src.models WARNING  google/gemma-2-27b-it not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory
2025-11-17 18:52:08 urllib3.connectionpool DEBUG    Starting new HTTPS connection (1): huggingface.co:443


2025-11-17 18:52:08 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /google/gemma-2-27b-it/resolve/main/config.json HTTP/1.1" 200 0
2025-11-17 18:52:08 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /google/gemma-2-27b-it/resolve/main/tokenizer_config.json HTTP/1.1" 200 0
2025-11-17 18:52:08 urllib3.connectionpool DEBUG    https://huggingface.co:443 "GET /api/models/google/gemma-2-27b-it/tree/main/additional_chat_templates?recursive=False&expand=False HTTP/1.1" 404 64


Loading checkpoint shards: 100%|██████████| 12/12 [00:08<00:00,  1.37it/s]

2025-11-17 18:52:22 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /google/gemma-2-27b-it/resolve/main/generation_config.json HTTP/1.1" 200 0


2025-11-17 18:52:22 urllib3.connectionpool DEBUG    https://huggingface.co:443 "HEAD /google/gemma-2-27b-it/resolve/main/custom_generate/generate.py HTTP/1.1" 404 0
2025-11-17 18:52:22 src.models INFO     loaded model <google/gemma-2-27b-it> | size: 51931.626 MB | dtype: torch.bfloat16 | device: cuda:0


## Saving the selection data
> For baseline evaluation. So that every LM is evaluated on the same set of data.

In [30]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
n_distractors = (
    5  # number of distractors. total options = n_distractors + 1 for SingleOne task
)
##########################################################

# symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
symantic_type = "landmarks"

TASK_CLS = SelectOneTask

select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)
select_task.categories

['name', 'prompt_templates', 'categories']


['United States',
 'France',
 'Italy',
 'United Kingdom',
 'Egypt',
 'India',
 'China',
 'Australia',
 'Brazil',
 'Japan',
 'Greece',
 'Mexico',
 'Peru',
 'Spain']

In [32]:
sample = select_task.get_random_sample(
    mt=mt,
    option_style=option_style,
    prompt_template_idx=prompt_template_idx,
    category="Japan",
    filter_by_lm_prediction=False,
)

print(sample.prompt(), ">>")
print(f'"{mt.tokenizer.decode([sample.ans_token_id])}"')

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

Options: Khajuraho Temples, Acropolis/Parthenon, Shanghai Skyline/The Bund, Colosseum, Golden Pavilion (Kinkaku-ji), Champs-Élysées.
Which of these landmarks is in Japan?
Answer: >>
" Golden"


[[PredictedToken(token=' Golden', prob=0.82421875, logit=19.625, token_id=17489, metadata=None),
  PredictedToken(token=' **', prob=0.1259765625, logit=17.75, token_id=5231, metadata=None),
  PredictedToken(token='  ', prob=0.031982421875, logit=16.375, token_id=139, metadata=None),
  PredictedToken(token='\n', prob=0.0031585693359375, logit=14.0625, token_id=108, metadata=None),
  PredictedToken(token=' ', prob=0.0027923583984375, logit=13.9375, token_id=235248, metadata=None)]]

In [33]:
import random
random.randint(1,3)

3

In [34]:
from tqdm.auto import tqdm

################################################################################################
LIMIT = 1024
N_DISTRACTORS = 5
DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)
################################################################################################

os.makedirs(DS_ROOT, exist_ok=True)

evaluation_samples = []
for _ in tqdm(range(LIMIT)):
    sample = select_task.get_random_sample(
        mt=mt,
        option_style=option_style,
        prompt_template_idx=prompt_template_idx,
        n_distractors=N_DISTRACTORS,
        # n_options=random.randint(1, 3),
        filter_by_lm_prediction=False,
    )
    evaluation_samples.append(sample)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "w") as f:
    json.dump(
        [sample.to_dict() for sample in evaluation_samples],
        f,
        indent=4,
    )

  0%|          | 0/1024 [00:00<?, ?it/s]

100%|██████████| 1024/1024 [00:11<00:00, 91.38it/s]


## Load Evaluation Samples

In [5]:
from src.selection.data import (
    SelectOneTask,
    SelectFirstTask,
    SelectLastTask,
    YesNoTask,
    CountingTask,
)
from typing import Literal
from src.selection.data import SelectionSample, YesNoSample, CountingSample
from src.selection.utils import get_first_token_id
from src.selection.data import MCQify_sample, COUNT_STR_MAP

##########################################################
prompt_template_idx = 3  # try out different templates
option_style: Literal["single_line", "numbered"] = "single_line"
# symantic_type = "objects"
# symantic_type = "profession"
# symantic_type = "nationality"
symantic_type = "landmarks"
##########################################################

TASK_CLS = SelectOneTask
select_task = TASK_CLS.load(
    path=os.path.join(env_utils.DEFAULT_DATA_DIR, "selection", f"{symantic_type}.json")
)

DS_ROOT = os.path.join(
    env_utils.DEFAULT_RESULTS_DIR, "selection/baseline", select_task.task_name
)

with open(os.path.join(DS_ROOT, f"{symantic_type}.json"), "r") as f:
    raw_samples = json.load(f)

if TASK_CLS == YesNoTask:
    evaluation_samples = [YesNoSample.from_dict(d) for d in raw_samples]
elif TASK_CLS == CountingTask:
    evaluation_samples = [CountingSample.from_dict(d) for d in raw_samples]
else:
    evaluation_samples = [SelectionSample.from_dict(d) for d in raw_samples]

prompt_template = select_task.prompt_templates[prompt_template_idx]

for idx in range(len(evaluation_samples)):
    evaluation_samples[idx].prompt_template = prompt_template
    # evaluation_samples[idx].option_style = option_style
    if isinstance(evaluation_samples[idx], SelectionSample):
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=evaluation_samples[idx].answer, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], CountingSample):
        count_str = COUNT_STR_MAP[evaluation_samples[idx].count]
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            name=count_str, tokenizer=mt.tokenizer, prefix=" "
        )
    elif isinstance(evaluation_samples[idx], YesNoSample):
        yes_mode = evaluation_samples[idx].yes
        evaluation_samples[idx].ans_token_id = get_first_token_id(
            "Yes" if yes_mode else "No", tokenizer=mt.tokenizer, prefix=" "
        )
    # evaluation_samples[idx] = MCQify_sample(
    #     sample=evaluation_samples[idx], tokenizer=mt.tokenizer
    # )

sample = evaluation_samples[15]
print(sample.prompt(), ">>", f'"{mt.tokenizer.decode(sample.ans_token_id)}"')

['name', 'prompt_templates', 'categories']
Options: Chichen Itza, Great Barrier Reef, Pompeii, Manu National Park, Plaza de Armas Cusco, Guanajuato City.
Which of these landmarks is in Australia?
Answer: >> " Great"


In [6]:
from src.selection.utils import get_first_token_id, verify_correct_option
from src.selection.data import get_options_for_answer

result = verify_correct_option(
    mt=mt,
    target=sample.ans_token_id,
    options=get_options_for_answer(sample),
    input=sample.prompt(),
    k=10,
)
result

(True,
 [PredictedToken(token=' Great', prob=0.94140625, logit=21.5, token_id=6553, metadata=None),
  PredictedToken(token=' **', prob=0.05322265625, logit=18.625, token_id=5231, metadata=None),
  PredictedToken(token=' The', prob=0.0031890869140625, logit=15.8125, token_id=714, metadata=None),
  PredictedToken(token='  ', prob=0.001251220703125, logit=14.875, token_id=139, metadata=None),
  PredictedToken(token='\n\n', prob=0.0004596710205078125, logit=13.875, token_id=109, metadata=None),
  PredictedToken(token=' ', prob=6.628036499023438e-05, logit=11.9375, token_id=235248, metadata=None),
  PredictedToken(token='\n', prob=6.628036499023438e-05, logit=11.9375, token_id=108, metadata=None),
  PredictedToken(token='   ', prob=2.586841583251953e-05, logit=11.0, token_id=140, metadata=None),
  PredictedToken(token=' *', prob=2.014636993408203e-05, logit=10.75, token_id=649, metadata=None),
  PredictedToken(token='\n\n\n', prob=1.895427703857422e-05, logit=10.6875, token_id=110, metadata

In [7]:
from tqdm import tqdm

results = []
for sample in tqdm(evaluation_samples):
    is_correct, pred, track = verify_correct_option(
        mt=mt,
        target=sample.ans_token_id,
        options=get_options_for_answer(sample),
        input=sample.prompt(),
    )
    results.append(
        {
            "sample": sample,
            "is_correct": is_correct,
            "predicted_option": pred,
            "track": track,
        }
    )

  0%|          | 0/1024 [00:00<?, ?it/s]

100%|██████████| 1024/1024 [01:34<00:00, 10.78it/s]


In [9]:
import numpy as np

ranks = []
logits = []
for result in results:
    sample = result["sample"]
    cur_rank = result["track"][sample.ans_token_id][0]
    ranks.append(cur_rank)
    logits.append(result["track"][sample.ans_token_id][1].logit)

n_correct = sum([1 for result in results if result["is_correct"]])
accuracy = n_correct / len(results)

ranks = np.array(ranks)
ranks_avg = ranks.mean()
ranks_std = ranks.std()

logits = np.array(logits)
logits_avg = logits.mean()
logits_std = logits.std()

print(
    f"Accuracy: {accuracy*100:.2f}% ({n_correct}/{len(results)}) | Avg. Rank: {ranks_avg:.2f} ± {ranks_std:.2f} | Avg. Logit: {logits_avg:.2f} ± {logits_std:.2f}"
)

Accuracy: 98.14% (1005/1024) | Avg. Rank: 1.30 ± 4.32 | Avg. Logit: 21.18 ± 1.38


In [10]:
print(sample.prompt())

Options: Fraser Island, Abu Simbel Temples, Shibuya Crossing, Golden Temple, Santiago de Compostela Cathedral, Teatro Amazonas.
Which of these landmarks is in Egypt?
Answer:


In [11]:
failed_cases = [result for result in results if not result["is_correct"]]

In [42]:
# sample = failed_cases[26]["sample"]
# print(sample.prompt())

In [43]:
from src.functional import predict_next_token

predict_next_token(
    inputs=sample.prompt(),
    mt=mt,
)

[[PredictedToken(token=' Abu', prob=0.90234375, logit=19.75, token_id=19556, metadata=None),
  PredictedToken(token=' **', prob=0.07421875, logit=17.25, token_id=5231, metadata=None),
  PredictedToken(token='  ', prob=0.01373291015625, logit=15.5625, token_id=139, metadata=None),
  PredictedToken(token='Abu', prob=0.002105712890625, logit=13.6875, token_id=64093, metadata=None),
  PredictedToken(token=' ', prob=0.000823974609375, logit=12.75, token_id=235248, metadata=None)]]

In [27]:
sample.ans_token_id, mt.tokenizer.decode(sample.ans_token_id)

(27179, ' Pope')